# NeuroAssist fine-tune 5/5 — Speech-to-text (Whisper small)

Fine-tunes **openai/whisper-small** on LibriSpeech **train-clean-100**, then converts it to the CTranslate2 format that the app's faster-whisper loads.

The practice recordings come from LibriSpeech *test-clean*, so this notebook never touches test-clean. It scores on *validation-clean* (word error rate, lower is better).

**Expect a small gain at best.** Stock Whisper small already makes only about 3–4 word errors per 100 here. LibriSpeech transcripts are uppercase with no punctuation, so they're converted to sentence case; the model may still lose some punctuation. Keep it only if the WER drops **and** transcripts still read well.

## 1. Install

In [ ]:
!pip install -q -U transformers datasets accelerate jiwer soundfile ctranslate2 faster-whisper

## 2. ⚙️ Settings

In [ ]:
BASE_MODEL  = "openai/whisper-small"
RUN_NAME    = "neuro-whisper-small"
APP_SETTING = "NEUROASSIST_WHISPER_MODEL"

TRAIN_ROWS  = 6000           # utterances from LibriSpeech clean/train.100 (28,539 available)
EVAL_ROWS   = 400            # from clean/validation (never test: the practice audio is test-clean)
MAX_STEPS   = 1000           # ~2.7 epochs over 6,000 clips at batch 16
LEARNING_RATE = 1e-5
BATCH_SIZE  = 16
WARMUP_STEPS = 100
EVAL_EVERY  = 250
SEED = 3407
SAVE_TO_DRIVE, DRIVE_DIR = True, "/content/drive/MyDrive/neuroassist-finetunes"

## 3. GPU and Google Drive

In [ ]:
import torch, os, io, json, random
import numpy as np
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))
random.seed(SEED)
OUT_DIR = f"/content/{RUN_NAME}"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = f"{DRIVE_DIR}/{RUN_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)

## 4. Load LibriSpeech (train-clean-100 and validation-clean)

In [ ]:
import soundfile as sf
from datasets import load_dataset, Audio

def sentence_case(t):
    t = t.strip().lower()
    t = re.sub(r"\bi\b", "I", t)
    return (t[:1].upper() + t[1:] + ".") if t else t

import re
def librispeech(split, n, shuffle):
    ds = load_dataset("openslr/librispeech_asr", "clean", split=split, streaming=True)
    ds = ds.cast_column("audio", Audio(decode=False))       # decode ourselves: no extra codec packages
    if shuffle:
        ds = ds.shuffle(seed=SEED, buffer_size=2000)
    rows = []
    for r in ds.take(n):
        a, sr = sf.read(io.BytesIO(r["audio"]["bytes"]), dtype="float32")
        if a.ndim > 1:
            a = a.mean(1)
        if sr != 16000:
            a = np.interp(np.arange(0, len(a) * 16000 / sr) * sr / 16000, np.arange(len(a)), a).astype(np.float32)
        rows.append({"audio": a, "text": sentence_case(r["text"])})
    return rows

train_rows = librispeech("train.100", TRAIN_ROWS, shuffle=True)
eval_rows = librispeech("validation", EVAL_ROWS, shuffle=False)
print(len(train_rows), "train ·", len(eval_rows), "eval ·", "example:", train_rows[0]["text"][:80])

## 5. Model, processor and batching

In [ ]:
from dataclasses import dataclass
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="English", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.language = "english"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False

class Rows(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

@dataclass
class Collator:
    def __call__(self, batch):
        feats = processor.feature_extractor([b["audio"] for b in batch], sampling_rate=16000,
                                            return_tensors="pt").input_features
        lab = processor.tokenizer([b["text"] for b in batch], padding=True, return_tensors="pt")
        labels = lab.input_ids.masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:, 0] == model.config.decoder_start_token_id).all():
            labels = labels[:, 1:]
        return {"input_features": feats, "labels": labels}

import jiwer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
normalize = BasicTextNormalizer()

def compute_metrics(pred):
    ids, out = pred.label_ids, pred.predictions
    ids[ids == -100] = processor.tokenizer.pad_token_id
    out[out == -100] = processor.tokenizer.pad_token_id
    hyp = processor.tokenizer.batch_decode(out, skip_special_tokens=True)
    ref = processor.tokenizer.batch_decode(ids, skip_special_tokens=True)
    pairs = [(normalize(r), normalize(h)) for r, h in zip(ref, hyp) if normalize(r).strip()]
    return {"wer": jiwer.wer([r for r, _ in pairs], [h for _, h in pairs])}

## 6. Train (with word error rate before and after)

In [ ]:
import inspect
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

wanted = dict(output_dir=f"/content/checkpoints/{RUN_NAME}", per_device_train_batch_size=BATCH_SIZE,
              per_device_eval_batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS,
              max_steps=MAX_STEPS, gradient_checkpointing=True, bf16=torch.cuda.is_bf16_supported(),
              fp16=not torch.cuda.is_bf16_supported(), eval_strategy="steps", eval_steps=EVAL_EVERY,
              predict_with_generate=True, generation_max_length=225, logging_steps=25, save_strategy="no",
              report_to="none", remove_unused_columns=False, seed=SEED)
accepted = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
args = Seq2SeqTrainingArguments(**{k: v for k, v in wanted.items() if k in accepted})
kw = dict(model=model, args=args, train_dataset=Rows(train_rows), eval_dataset=Rows(eval_rows),
          data_collator=Collator(), compute_metrics=compute_metrics)
kw["processing_class" if "processing_class" in inspect.signature(Seq2SeqTrainer.__init__).parameters
   else "tokenizer"] = processor.feature_extractor
trainer = Seq2SeqTrainer(**kw)

before = trainer.evaluate()["eval_wer"]
print(f"WER before: {before:.4f}")
trainer.train()
after = trainer.evaluate()["eval_wer"]
print(f"WER before {before:.4f} → after {after:.4f}  ({'better' if after < before else 'NOT better: keep stock Whisper'})")
json.dump({"model": BASE_MODEL, "wer_before": before, "wer_after": after}, open(f"{OUT_DIR}/scores.json", "w"), indent=2)

## 7. Convert for faster-whisper and test it

In [ ]:
import subprocess, shutil
merged = f"/content/{RUN_NAME}_hf"
model.save_pretrained(merged); processor.save_pretrained(merged)
ct2 = f"/content/{RUN_NAME}_ct2"
shutil.rmtree(ct2, ignore_errors=True)
r = subprocess.run(["ct2-transformers-converter", "--model", merged, "--output_dir", ct2, "--quantization", "int8",
                    "--copy_files", "tokenizer.json", "preprocessor_config.json"], capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
assert r.returncode == 0, "Conversion failed (see log). The Hugging Face copy is still saved below."

from faster_whisper import WhisperModel
fw = WhisperModel(ct2, device="cpu", compute_type="int8")     # same settings as the app
sf.write("/content/sample.wav", eval_rows[0]["audio"], 16000)
segs, _ = fw.transcribe("/content/sample.wav", beam_size=5, vad_filter=True, language="en")
print("REF:", eval_rows[0]["text"]); print("OUT:", " ".join(s.text.strip() for s in segs))

shutil.copytree(ct2, f"{OUT_DIR}/ct2", dirs_exist_ok=True)
shutil.copytree(merged, f"{OUT_DIR}/hf", dirs_exist_ok=True)
print(f"""
Saved to {OUT_DIR}/ct2. On the demo laptop, copy that folder (e.g. to C:\\models\\{RUN_NAME}) and set:
    $env:{APP_SETTING}="C:\\models\\{RUN_NAME}"
""")